# Dekódolási stratégiák nyelvi modellekhez

Minden dekódolási stratégiához szükséges egy $\theta$ autoregresszív nyelvi modell (language model).

A modell bemenete ún. tokenek sorozatából áll, amik szavakat, szórészeket, esetleg bájtokat reprezentálnak és a $V=\{0,1, \ldots, n-1\} \in \mathbb{N}_0$ véges halmazra vannak leképezve. A halmazt a modell szótárának nevezik és $|V|=n$ a modell szótárának mérete.

A bemenet alakja $B \times S$, ahol $B$ a batch-méret (pl. 512) és $S$ a szekvencia hossz (pl. 8192). Más szavakkal, a bemenet $B$ darab, egyenként $S$ hosszú token sorozata (vagy más néven szekvenciája).

A modell kimenete azonban nem egyetlen token, hanem egy diszkrét valószínűségi eloszlás a szótár felett. Egyszerűbben: a szótár minden eleméhez egy valószínűság van rendelve, aminek összege 1 (még további tulajdonságok is teljesülnek, de ezek jelenleg nem lényegesek). Az kimenet alakja
$B \times n$, azaz egy adott szekvencia következő elemére egy eloszlást kapunk.

A autóregresszív modellek az ún. "következő-token" elvét követik.

Egy adott szekvencia esetén a következő token feltételes valószínűsége

$$
p_\theta\left(x_i \mid x_1, \ldots, x_{i-1} \right).
$$

Egy $m$ hosszú $\{x_1, x_2, \ldots, x_m\}$ szekvencia valószínűsége a feltételes valószínűségek szorzata, azaz

$$p_\theta(x)=p_\theta\left(x_1\right) \prod_{i=2}^m p_\theta\left(x_i \mid x_1, \ldots, x_{i-1}\right)
$$

A fenti képletek a modell általános leírását adják, amely a következőképpen
ragadható meg intuitívan. Minden lépésben a modell nem egy "kész" választ ad,
hanem egy diszkrét valószínűségi eloszlást: megmondja, mennyire valószínű, hogy
a következő token éppen egy megadott szótár elem lesz.

Felmerül a kérdés, hogy egy tetszőleges szekvencia esetén hogyan választunk
ebből az eloszlásból egyetlen konkrét tokent, a szekvencia következő elemét.

A válasz nem magától értetődő, ezért különböző technikák léteznek és mindegyik szöveget eredményezhet ugyanabból a modellből.

Az ehhez kapcsolódó döntéseket **dekódolási stratégiának** nevezik, ami egy megadott algoritmus (és/vagy leképezés) segítségével a modell által adott valószínűségi eloszlásból választ egy következő tokent. Ezt a lépést ismételve áll elő a teljes legenerált szekvencia.

A jelen notebook célja a különböző stratégiák áttekintése (a teljesség igénye nélkül).

Megjegyzés. Mint említettük, a teljes generált szekvencia lépésről lépésre áll elő: a választott tokent hozzáfűzzük a meglévő szekvenciához, a modellt újra lefuttatjuk, és megismételjük a választást egy leállási feltételig, ami lehet egy speciális token (end-of-sequence token, röviden EOS token) vagy a maximális szekvencia hossz elérése, ami egy előre megadott szám.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
# !hf auth login

In [3]:
model_id = "LiquidAI/LFM2-350M" # ~350 million parameters

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model = model.eval()

In [5]:
example_text = "Intelligence is an agent's ability to achieve goals in a wide range of environments."

In [6]:
example_tokens = tokenizer(example_text, return_tensors="pt")

In [7]:
input_ids = example_tokens["input_ids"]

In [8]:
input_ids, input_ids.shape

(tensor([[    1,  6286, 21482,   856,   902, 11380,  1090,  4978,   811,  8342,
           8154,   797,   768,  6266,  3795,   803, 13423,   523]]),
 torch.Size([1, 18]))

In [9]:
outputs = model.forward(input_ids)

In [10]:
outputs.logits.shape # (batch_size, seq_len, vocab_size)

torch.Size([1, 18, 65536])

In [11]:
outputs.logits[:, -1, :]

tensor([[-4.6250,  0.1221,  2.4062,  ..., -4.6250, -4.6250, -4.6250]],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [12]:
outputs.logits[:, -1, :].shape

torch.Size([1, 65536])

A fenti kódból látható, hogy a batch-méret $1$ és a szekvencia hossza $17$. Az `outputs.logits[:, -1, :]` a 18. tokenre vonatkozó ún. logitokból (számokból) álló vektor. Ez még nem valószínűségi eloszlás, mert nem normalizált. Ezt azt jelenti, hogy például az összegük lehet nagyobb mint 1 vagy a logitok lehetnek negatívak is (de a valószínűségek már nem).

Ahhoz, hogy diszkrét valószínűségi eloszlás kapjunk, a nyelvi modell kimenetére softmax függvényt szokás alkalmazni:

$$
\operatorname{Softmax}\left(x_i\right)=\frac{\exp \left(x_i\right)}{\sum_j \exp \left(x_j\right)}
$$

A függvény a szekvencia elemeit (a logitokat) egy adott dimenzió mentén a $[0,1]$ zárt intervallumba képezi le, aminek összege 1 lesz.

In [13]:
results = torch.nn.functional.softmax(outputs.logits, dim=-1)

In [14]:
results[:, -1, :]

tensor([[2.5757e-09, 2.9616e-07, 2.9057e-06,  ..., 2.5757e-09, 2.5757e-09,
         2.5757e-09]], dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [15]:
results[:, -1, :].sum()

tensor(1., dtype=torch.bfloat16, grad_fn=<SumBackward0>)

In [16]:
logits = results[:, -1, :]
logits, logits.shape

(tensor([[2.5757e-09, 2.9616e-07, 2.9057e-06,  ..., 2.5757e-09, 2.5757e-09,
          2.5757e-09]], dtype=torch.bfloat16, grad_fn=<SelectBackward0>),
 torch.Size([1, 65536]))

Megjegyzés. A softmax függvényt általában a modellen belül implementálják.

Vezessünk be néhány jelölést.

Legyen $\theta$ modell (vagy pontosabban $\theta$ a modell paraméterei), $V$ a szótár, melynek mérete $|V| = n$.

A modell a következő tokenre egy diszkrét valószínűségi eloszlást ad. Jelölje ezt

$$
p_\theta(\cdot \mid x_1, x_2, \ldots, x_{i-1})
$$

ahol a $\cdot$ jel az egész eloszlásra utal.

A szótár egy tetszőleges elemének valószínűsége (adott szekvencia és modell mellett)

$$
p_\theta(v \mid x_1, x_2, \ldots, x_{i-1}) \quad v \in V.$$


A dekódolási stratégia által kiválasztott token valószínűsége (adott szekvencia és modell mellett):

$$
p_\theta(x_i \mid x_1, x_2, \ldots, x_{i-1}).
$$


## Hőmérséklet (temperature)

A hőmérséklet (temperature) nem egy dekódolási stratégia, hanem egy külön lépés, amely során a softmax függvény alkalmazása előtt a logitokból álló vektorokat egy $t > 0$ paraméterrel osztjuk.

 A $t$ azt szabályozza, mennyire "éles" vagy "lapos" az eloszlás.

Ha $0 < t < 1$, akkor az eloszlás "élesebb" lesz: a valószínűség a legnagyobb logitú tokenek köré tömörül. A modell a legvalószínűbb tokeneket fogja előnyben részesíteni.

Ha $t > 1$, akkor az eloszlás "laposabb" lesz: a kisebb valószínűségű tokenek is nagyobb eséllyel kerülnek be. A modell kreatívabb, változatosabb, de
egyúttal összefüggéstelenebb is lehet.

Ha $t = 1$, akkor az eloszlás nem változik, azaz nincs hatása az eloszlásra.

Két speciális vagy határeset van. Ha $t \to 0$ (olvasd: ha $t$ tart a nullához), akkor a választás egyetlen tokenre koncentrálódik, a legnagyobb értékű (vagy valószínűségű) tokenre. Ha $t \to \infty$ (olvasd: ha $t$ tart a végtelenhez), akkor a logitok közötti különbségek eltűnnek és az eloszlás egyenletes lesz.

Egy minimális példa:

In [17]:
logits = torch.tensor([2.0, 1.0, 0.5, 0.25, 0.1])

In [18]:
t1 = 0.0001
t2 = 1000

logits / t1, logits / t2

(tensor([20000.0000, 10000.0000,  5000.0000,  2500.0000,  1000.0001]),
 tensor([0.0020, 0.0010, 0.0005, 0.0003, 0.0001]))

## Mohó dekódolás (greedy decoding)

A legegyszerűbb dekódolási stratégia a mohó dekódolás. A stratégia minden
lépésben azt a tokent választja a $V$ szótár összes eleme közül, amelyhez a modell a legnagyobb valószínűséget rendeli. Formálisan:


$$
x_i = \operatorname*{argmax}_{v \in V} p_\theta\left(v \mid x_1, \ldots, x_{i-1} \right)
$$


In [19]:
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
) -> torch.Tensor:

    for _ in range(max_new_tokens):

        outputs = model.forward(inputs)
        logits = outputs.logits

        # last time step and temperature scaling
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = logits[:, -1, :] / temperature

        # probability of each token in vocabulary
        probs = torch.softmax(logits, dim=-1)

        # get the idx of the vocab entry with the highest prob value
        idx_next = torch.argmax(probs, dim=-1, keepdim=True)  # (batch, 1)

        # stop at end-of-sequence token (e.g. the number 7)
        if idx_next == tokenizer.eos_token_id:
            break

        inputs = torch.cat((inputs, idx_next), dim=1)

    return inputs

In [20]:
results = generate(input_ids)

In [21]:
tokenizer.decode(results)

["<|startoftext|>Intelligence is an agent's ability to achieve goals in a wide range of environments. Intelligence is often measured by the ability to solve problems, make decisions, and understand complex concepts. It can be categorized into several types, including analytical, creative, and practical intelligence.\n\n### Analytical Intelligence\nAnalytical intelligence involves the ability to break"]

## Multinomiális mintavételezés

A multinomiális mintavételezés véletlenszerűen választ egy token az adott diszkrét valószínűségi eloszlásból, pontosan akkora valószínűséggel, amit a modell hozzá rendelt, ezért minden tokennek lehet esélye arra, hogy kiválasztásra kerüljön. Ezért ez a stratégia nemdeterminisztikus.

Formálisan, az

$$
x_i \sim p_\theta(\cdot \mid x_1, \ldots, x_{i-1} ).
$$

Itt a $\sim$ jel azt jelenti, hogy "az eloszlásból mintavételezünk": $x_i$ egy
valószínűségi változó, amely egy tetszőleges $v$ szótárbeli értékét $p_\theta(v \mid x_1, \ldots, x_{i-1})$ valószínűséggel veszi fel.

In [22]:
@torch.no_grad()
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
) -> torch.Tensor:

    for _ in range(max_new_tokens):

        outputs = model.forward(inputs)
        logits = outputs.logits

        # last time step and temperature scaling
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = logits[:, -1, :] / temperature

        # probability of each token in vocabulary
        probs = torch.softmax(logits, dim=-1)

        # get the idx of the vocab entry by multinomial sampling
        idx_next = torch.multinomial(probs, num_samples=1)

        # stop at end-of-sequence token
        if idx_next == tokenizer.eos_token_id:
            break

        inputs = torch.cat((inputs, idx_next), dim=1)

    return inputs

In [23]:
results = generate(input_ids)

In [24]:
tokenizer.decode(results)

["<|startoftext|>Intelligence is an agent's ability to achieve goals in a wide range of environments. It is a crucial element in decision-making and problem-solving, as well as in the overall performance of an individual. Understanding intelligence can help in various contexts, from education and workplace productivity to personal development and leadership.\n\n**Types of Intelligence:**\n\n1"]

Megjegyzés. A hőmérséklet jelentősége itt már tisztán látszik. A multinomiális mintavételezés tehát egy eloszlásból választ, amelyet a hőmérséklet alakít / skáláz.

## Top-k mintavételezés

A top-k mintavételezés során a $k$ legvalószínűbb tokent vesszük figyelembe, nem a teljes $V$ szótárt. Ezáltal a valószínűtlen tokenek egyáltalán nem kerülhetnek be a generált szekvenciába. A módszer továbbra is nemdeterminisztikus, mert a megmaradt $k$ token közül véletlenszerűen választ.


Jelölje $\{v_1, v_2, \ldots, v_k\} = W \subseteq V$ azt a $k$ tokent, amelyekhez a legnagyobb valószínűségek tartoznak. A $W-V$ halmaz elemeit, vagyis a többi token valószínűségét nullára állítjuk:

$$
p'_\theta(v \mid x_1, x_2, \ldots, x_{i-1}) =
\begin{cases}
p_\theta(v \mid x_1, \ldots, x_{i-1}) & \text{ha } v \in W, \\[2mm]
0 & \text{egyébként.}
\end{cases}
$$

A nem-nulla valószínűségű tokenekre sotfmax függvényt alkalmazzuk és
a következő token a szűkített eloszlásból kerül véletlenszerűen kiválasztásra:

$$
x_i \sim p'_\theta(\cdot \mid x_1, \ldots, x_{i-1} ).
$$

A probléma a top-k mintavételezéssel, hogy a $k$ paraméter optimális értéke más lehet a különböző modellek és generálások között is. Amikor a modell biztos a válaszában (pl. egy ismétlődő szekvencia generálása esetén), akkor az első néhány token elegendő információt hordozhat. Ha bizonytalan, akkor az alacsony $k$ érték korlátozhatja a megfelelő választ.

In [25]:
@torch.no_grad()
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
    top_k: int = 3
) -> torch.Tensor:

    for _ in range(max_new_tokens):

        outputs = model.forward(inputs)
        logits = outputs.logits

        # last time step and temperature scaling
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = logits[:, -1, :] / temperature

        top_logits, top_pos = torch.topk(logits, top_k)

        # select top k possible tokens, assign -inf to all others in batch
        logits = torch.where(
            condition=logits < top_logits[:, [-1]],
            input=torch.tensor(float('-inf')),
            other=logits
        )
        # probability of each token in vocabulary
        probs = torch.softmax(logits, dim=-1)

        # get the idx of the vocab entry by multinomial sampling
        idx_next = torch.multinomial(probs, num_samples=1)

        # stop at end-of-sequence token
        if idx_next == tokenizer.eos_token_id:
            break

        inputs = torch.cat((inputs, idx_next), dim=1)

    return inputs

In [26]:
results = generate(input_ids)

In [27]:
tokenizer.decode(results)

["<|startoftext|>Intelligence is an agent's ability to achieve goals in a wide range of environments. In psychology, intelligence is often measured by IQ tests. However, intelligence encompasses many different aspects, including cognitive abilities, emotional intelligence, and practical skills.\n\n### Types of Intelligence\n\n1. **Verbal Intelligence**: The ability to understand and use"]

## Top-p mintavételezés

A top-p mintavételezés (top-p sampling, nucleus sampling) a legkisebb token halmazból választ véletlenszerűen, mely halmaz kumulatív valószínűsége meghalad egy $p$ küszöbértéket - egy 0 és 1 közötti értéket.

Például, ha $p=0.9$, a modell a legkisebb számú tokent veszi figyelembe, amelynek összesített valószínüsége $90 \%$.

Legyen $W \subseteq V$ a legkisebb olyan halmaz, amire

$$
\sum_{v \in W} p_{\theta}\left(v \mid x_1, \ldots, x_{i-1} \right) \geq p
$$

teljesül. A halmaz elemeire alkalmazzuk a softmax függvényt és a véletlenszerű mintavételezés ezután történik.


A módszer segíthet elkerülni a repetitív és természetellenes szövegek generálásának problémáját.

In [28]:
@torch.no_grad()
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
    top_p: float = 0.9
) -> torch.Tensor:

    assert 0.0 < top_p < 1.0, "top_p must be between 0 and 1."

    for _ in range(max_new_tokens):

        outputs = model.forward(inputs)
        logits = outputs.logits

        # last time step and temperature scaling
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = logits[:, -1, :] / temperature

        # probability of each token in vocabulary
        probs = torch.softmax(logits, dim=-1)

        # sort probabilities in descending order
        probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)

        # create cumulative sum of elements
        probs_sum = torch.cumsum(probs_sort, dim=-1)

        # mark tokens having values over top_p
        mask = probs_sum - probs_sort > top_p
        probs_sort[mask] = 0.0

        # renormalize remaining probabilities
        probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))

        # get the idx of the probabilites by multinomial sampling
        idx_next = torch.multinomial(probs_sort, num_samples=1)

        # get original index
        idx_next = torch.gather(probs_idx, -1, idx_next)

        # stop at end-of-sequence token
        if idx_next == tokenizer.eos_token_id:
            break

        inputs = torch.cat((inputs, idx_next), dim=1)

    return inputs

In [29]:
results = generate(input_ids)

In [30]:
tokenizer.decode(results)

["<|startoftext|>Intelligence is an agent's ability to achieve goals in a wide range of environments. Intelligence is usually measured in terms of the ability to solve problems, think logically, and understand complex concepts. Intelligence can also be broken down into several different components, including verbal reasoning, mathematical reasoning, spatial reasoning, and working memory.\n\nTheories"]

## Nyaláb keresés (beam search)

Ashwin K. Vijayakumar et al. 2016. Diverse Beam Search: Decoding Diverse Solutions from Neural Sequence Models. https://arxiv.org/abs/1610.02424

A nyalábkeresés (beam search) minden egyes (idő)lépésben, vagyis minden egyes token generálásakor egyszerre több jelölt szekvenciát, ún. nyalábot tart fenn, és ezeket párhuzamosan bővíti, így egyszerre több utat vizsgálunk. Ehhez szükséges egy $b$ paraméter, a nyaláb szélessége (vagy nyalábok száma).

A modell kimenetére softmax függvényt alkalmazva egy valószínűségi eloszlást
kapunk a szótár felett. Az eloszlás elemeinek vesszük a (természetes)
logaritmusát.

Az első lépésben kiválasztjuk a $b$ legnagyobb log-valószínűségű
tokent, és mindegyiket külön-külön hozzáfűzzük a bemenethez - így $b$ darab
jelölt szekvenciánk (nyalábunk) lesz.

Jelölje ismét $\{v_1, v_2, \ldots, v_b\} = W \subseteq V$ azt a $b$ tokent, amelyekhez a legnagyobb log-valószínűségek tartoznak. Ezek külön nyalábokként kezelendőek és mindegyik hozzáfűzésre kerül a szekvenciához

$$
\begin{aligned}
\mathbf{y}_1 &= (y_1, \ldots, y_t, v_1), \\
\mathbf{y}_2 &= (y_1, \ldots, y_t, v_2), \\
&\vdots \\
\mathbf{y}_b &= (y_1, \ldots, y_t, v_b).
\end{aligned}
$$

Ezeket együtt jelölje a $B_t = \{\mathbf{y}_1, \mathbf{y}_2, \ldots, \mathbf{y},_b\}, |B_t| = b$ halmaz a $t$-edik lépésben.

A kiválasztás az

$$
s(\mathbf{y}) = \sum_{j=1}^{t} \log p_\theta(y_j \mid y_1, y_2, \ldots, y_{j-1})
$$

képlet kiszámításával történik, ahol $y_1, y_2, \ldots, y_{t-1}$ a generált szekvencia (tokenek sorozata).

A következő lépésben a modell minden $B_t$ elemre számít egy új kimenetet, vagyis egy újabb valószínűségi eloszlást kapunk a $t+1$-edik lépésben és minden nyalábot bővítünk a szótár összes tokenjével, így $b \times n$ jelöltet kapunk. Ezt a $b \times n$ jelöltet egyetlen közös halmazba tesszük.

$$
C_{t+1} = \big\{\, (\mathbf{y}, v) \;\big|\; \mathbf{y} \in B_t,\ v \in V \,\big\},
\qquad |C_{t+1}| = b \times n.
$$

Mindegyik jelölthez kiszámításra kerül egy pontszám, a szekvencia tokenjeinek log-valószínűségeinek összege - a nyaláb korábbi pontszáma és az új token log-valószínűségének összege:

$$
s(\mathbf{y}) = s\big( (y_1, \ldots, y_t, v) \big) = s(y_1, \ldots, y_t) + \log p_\theta(v \mid y_1, \ldots, y_t),
$$

és globálisan megtartjuk a $b$ legnagyobb pontszámú páronként különböző nyalábot, függetlenül attól, melyik nyalábból származik.



A nyesés / kiválasztás lépésenként történik a nyalábok között (és nem a nyalábokon belül). Így egy eldobott nyaláb véglegesen elveszik, a keresés nem várja meg a végét, hogy lássa, milyen eredménye lett volna.

Ez egy közelítő keresés - nem garantálja a legnagyobb pontszámú
szekvencia megtalálását, cserébe a nyalábok száma minden lépésben konstans $b$
marad.

Tehát a $b$ paraméter egyszerre szabályozza a keresés alaposságát és a költségét: nagyobb $b$ több alternatíva között keres (így közelebb kerül a legjobb szekvenciához), de drágább.

Az eljárást addig ismételjük, amíg minden nyaláb EOS-tokent nem ér el, vagy el
nem érjük a maximális szekvenciahosszt (az EOS-t elért nyalábokat lezárjuk, de
megőrizésre kerül, mint kész jelölt). Végül a $b$ nyaláb közül a legnagyobb pontszámút adjuk vissza, azaz

$$
\mathbf{y}^{*} = \underset{\mathbf{y} \in B_t}{\arg\max}\; s(\mathbf{y}).
$$

A módszer hátránya, hogy a legtöbb folytatás egyetlen magasra értékelt nyalábból jön, ezért kevéssé változatás kimeneteket eredményezhet. Ennek mitigálására találták ki a diverz nyaláb keresést.

Megjegyzés. A nyaláb pontszáma azért log-valószínűségek összege és nem valószínűségek szorzata, mert az összeg számítási szempontból kedvezőbb (lásd [itt](https://cs.stackexchange.com/questions/77135/why-is-adding-log-probabilities-faster-than-multiplying-probabilities)) és a kettő matematikailag ekvivalens. A logaritmus azonossága miatt a

$$
\log\!\left( \prod_{j=1}^{t} p_\theta(x_j \mid x_1, x_2, \ldots, x_{j-1}) \right)
= \sum_{j=1}^{t} \log p_\theta(x_j \mid x_1, x_2, \ldots, x_{j-1}),
$$
azaz egy szekvencia valószínűségének logaritmusa a tokenenkénti log-valószínűségek összege.

A logaritmus szigorúan monoton nő, így sorrendet nem változtatja meg. Ezért, amelyik szekvenciának nagyobb a valószínűsége, annak nagyobb a log-valószínűsége is:
$$
p_\theta(y) \le p_\theta(y') \iff \log p_\theta(y) \le \log p_\theta(y').
$$


In [31]:
@torch.no_grad()
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
    num_beams: int = 3
) -> torch.Tensor:

    # each beam is a (score, sequence) pair start with a single
    # beam (input) the score is the sum of log-probabilities
    beams = [(0.0, inputs)]

    for _ in range(max_new_tokens):

        candidates = []

        for score, seq in beams:

            # a finished beam (ended with EOS) is kept, but not expanded
            if seq[0, -1].item() == tokenizer.eos_token_id:
                candidates.append((score, seq))
                continue

            outputs = model.forward(seq)
            logits = outputs.logits

            # last time step and temperature scaling
            # (batch, n_token, vocab_size) -> (batch, vocab_size)
            logits = logits[:, -1, :] / temperature

            # log-probabilities of each token in the vocabulary
            log_probs = torch.log_softmax(logits, dim=-1)

            # take the num_beams best continuations of current beam
            top_log_probs, top_idx = torch.topk(log_probs, num_beams, dim=-1)

            # add each continuation as a candidate
            for k in range(num_beams):
                next_token = top_idx[0, k].view(1, 1)
                next_score = score + top_log_probs[0, k].item()
                next_seq = torch.cat((seq, next_token), dim=1)
                candidates.append((next_score, next_seq))

        # keep best num_beams candidates across all beams
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:num_beams]

        # stop if every beam has finished
        if all(seq[0, -1].item() == tokenizer.eos_token_id for _, seq in beams):
            break

    # return the highest-scoring beam
    best_score, best_seq = max(beams, key=lambda x: x[0])
    return best_seq

In [32]:
results = generate(input_ids)

In [33]:
tokenizer.decode(results)

["<|startoftext|>Intelligence is an agent's ability to achieve goals in a wide range of environments. Intelligence can be defined as the capacity to reason, solve problems, learn, and adapt to new situations. It encompasses various cognitive abilities, including memory, attention, language, and mathematical reasoning. Intelligence is often measured using standardized tests, such as the We"]

## Büntetés (penalty) a dekódolási stratégiákban

### Ismétlés(ek) büntetése (repetition penalty)

Nitish Shirish Keskar et al. 2019. CTRL: A Conditional Transformer Language Model for Controllable Generation https://arxiv.org/abs/1909.05858

Az ismétlések büntetése során a már előfordult tokenek logitjait bűnteti egy adott szekvenciában. A büntetést egyszer alkalmazzuk egy időlépésben (függetlenül a korábbiaktól). Egy $r$ paraméterrel osztjuk a pozitív logitokat, és szorozzuk a negatív logitokat.

A dekódolási stratégia pedig tetszőleges lehet.

Legyen $\mathbf{z} = [ z_1, z_2, \ldots, z_n]$ a következő token kiválasztásához szükséges logitokból álló vektor és $Z$ az adott szekvencián belüli tokenek halmaza (ami a bemenet és lehet a már legenerált tokenek együttese).

A logitok módosítása a következő függvénnyel történik:

$$
z_i^{\prime} = \begin{cases}
\frac{y_i}{r} & \text { ha } i \in Z,  z_i>0 \\
z_i \cdot r & \text { ha } i \in Z, z_i \leq 0 \\
z_i & \text { ha } i \notin Z
\end{cases}
$$

In [34]:
@torch.no_grad()
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
    repetition_penality: float = 1.2,
) -> torch.Tensor:

    for _ in range(max_new_tokens):

        outputs = model.forward(inputs)
        logits = outputs.logits
        # last time step (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = logits[:, -1, :] / temperature

        batch_size = inputs.shape[0]
        for b in range(batch_size):

            used_tokens = inputs[b].unique()
            used_logits = logits[b, used_tokens]

            # apply repetition penalty
            logits[b, used_tokens] = torch.where(
                used_logits < 0,
                used_logits * repetition_penality,
                used_logits / repetition_penality
            )

        # probability of each token in vocabulary
        probs = torch.softmax(logits, dim=-1)

        # get the idx of the vocab entry by multinomial sampling
        idx_next = torch.multinomial(probs, num_samples=1)

        # stop at end-of-sequence token
        if idx_next == tokenizer.eos_token_id:
            break

        inputs = torch.cat((inputs, idx_next), dim=1)

    return inputs

In [35]:
results = generate(input_ids)

In [36]:
tokenizer.decode(results, skip_special_tokens=True)

["Intelligence is an agent's ability to achieve goals in a wide range of environments. Intelligence is measured by various factors, such as cognitive abilities (IQ scores), problem-solving skills, reasoning and learning capacity, emotional regulation, and adaptability.\nThe development of intelligence can be attributed to both genetic inheritance and environmental factors throughout one's life."]

### Gyakoriság büntetése (frequency penalty)

https://developers.openai.com/api/docs/guides/advanced-usage

A módszer egy adott szekvencia során a következő token kiválasztásához szükséges logitokból álló vektort arányosan bünteti a korábbi tokenek előfordulásával.

Legyen $\mathbf{z} = [z_1, z_2, \ldots, z_n]$ a következő token kiválasztásához
szükséges logitokból álló vektor.

Jelölje $c_i$ az $i$-edik token előfordulásainak számát a szekvenciában (a bemenet és a már generált tokenek együttesében). A módszer a logitokat az előfordulásuk számával arányosan módosítja:

$$
z_i' = z_i - c_i \cdot \alpha,
$$
ahol $\alpha \geq 0$ a büntetés erőssége.

Az adott szekvenciában nem szereplő tokenek nem módosítja, mert $c_i = 0$, ezért $y_i' = y_i$. A már előfordult tokenek logitjai pedig annál nagyobb mértékben csökken, minél többször szerepeltek.

In [37]:
@torch.no_grad()
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
    frequency_penalty: float = 0.9,
) -> torch.Tensor:

    for _ in range(max_new_tokens):

        outputs = model.forward(inputs)
        logits = outputs.logits
        # last time step (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = logits[:, -1, :] / temperature

        batch_size = inputs.shape[0]
        for b in range(batch_size):
            # apply frequency penalty
            counts = torch.bincount(inputs[b], minlength=logits.shape[-1])
            logits[b] -= counts * frequency_penalty

        # probability of each token in vocabulary
        probs = torch.softmax(logits, dim=-1)

        # get the idx of the vocab entry by multinomial sampling
        idx_next = torch.multinomial(probs, num_samples=1)

        # stop at end-of-sequence token
        if idx_next == tokenizer.eos_token_id:
            break

        inputs = torch.cat((inputs, idx_next), dim=1)

    return inputs

In [38]:
results = generate(input_ids)

In [39]:
tokenizer.decode(results, skip_special_tokens=True)

["Intelligence is an agent's ability to achieve goals in a wide range of environments. Intelligence encompasses various cognitive abilities, including reasoning, problem-solving, memory, perception, and learning. Different types of intelligence are identified through various theories such as the General Intelligence (g) model and the Multiple Intelligences theory by Howard Gardner.\n\n###"]

### Jelenlét büntetés (presence penalty)

https://developers.openai.com/api/docs/guides/advanced-usage

A jelenlét büntetése (presence penalty) a gyakoriság büntetéséhez hasonlóan additív, de nem az előfordulások számával arányosan büntet, hanem azt vizsgálja, hogy egy adott token előfordult-e már. Ha igen, akkor az adott logitból egy $\alpha$ értéket vonunk ki.

In [40]:
@torch.no_grad()
def generate(
    inputs: torch.Tensor, # (batch_size, num_tokens)
    max_new_tokens: int = 50,
    temperature: float = 0.6,
    presence_penalty: float = 0.9,
) -> torch.Tensor:

    for _ in range(max_new_tokens):

        outputs = model.forward(inputs)
        logits = outputs.logits
        # last time step (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = logits[:, -1, :] / temperature

        batch_size = inputs.shape[0]
        for b in range(batch_size):
            # apply presence penalty
            counts = torch.bincount(inputs[b], minlength=logits.shape[-1])
            logits[b] -= (counts > 0).float() * presence_penalty

        # probability of each token in vocabulary
        probs = torch.softmax(logits, dim=-1)

        # get the idx of the vocab entry by multinomial sampling
        idx_next = torch.multinomial(probs, num_samples=1)

        # stop at end-of-sequence token
        if idx_next == tokenizer.eos_token_id:
            break

        inputs = torch.cat((inputs, idx_next), dim=1)

    return inputs

In [41]:
results = generate(input_ids)

In [42]:
tokenizer.decode(results, skip_special_tokens=True)

["Intelligence is an agent's ability to achieve goals in a wide range of environments. Intelligence refers to a person's capacity to reason, learn, and solve problems. Intelligence is often measured by intelligence quotient (IQ) tests, which assess a person's ability to think logically, understand abstract concepts, and make decisions.\n\n###"]